# Checkpoint 2 — сохранить model outputs

Одна загрузка каждой модели: hidden states → raw behavioral logits → сохранение → release. Valid cache пропускает inference. Выберите kernel с зависимостями и доступом к весам Gemma. См. [README](README.md).

In [ ]:
from pathlib import Path
import sys

sys.dont_write_bytecode = True
PROJECT = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / "latent-behavior-alignment")
    if (candidate / "notebooks/checkpoint2/artifacts.py").is_file()
)
MODULE_DIR = PROJECT / "notebooks/checkpoint2"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
from artifacts import MODELS, load_dataset, output_root

DATASET = "mixed"
ARTIFACTS = output_root(DATASET) / "artifacts"
data = load_dataset(dataset=DATASET)
SELECTED_MODELS = list(MODELS)  # Or ["gemma-2-2b"] for the first run.
print("Dataset:", len(data), "train:", len(data.train_idx), "test:", len(data.test_idx))


## Проверка и построение cache

Модели идут последовательно. Instruct-модели сохраняют обычные и chat logits. Частичный/несовместимый cache вызывает явную ошибку; инструкции восстановления — в README.

In [ ]:
from inference import ensure_cache

for name in SELECTED_MODELS:
    ensure_cache(ARTIFACTS / name, MODELS[name], data)


## Готово для offline-анализа

После успешного выполнения можно закрыть kernel и открыть `02_offline_analysis.ipynb`. CCS probes создаются там, без основной модели.